# Recurrent Neural Networks

## Learning Objectives
1. Implement a vanilla RNN forward pass in numpy and demonstrate vanishing gradients.
2. Build a custom LSTM cell in PyTorch and train on sequence prediction.
3. Compare RNN vs LSTM vs GRU for long-range dependency learning.
4. Analyze gradient norms across timesteps to understand gating mechanisms.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## Level 1: Vanilla RNN and Vanishing Gradients (numpy)

The vanilla RNN update rule:
```
h_t = tanh(W_hh @ h_{t-1}  +  W_xh @ x_t  +  b)
```
During BPTT the gradient signal is multiplied by W_hh at each timestep.
If ||W_hh|| < 1 the gradient decays exponentially → **vanishing gradient**.
If ||W_hh|| > 1 it explodes → **exploding gradient**.

In [ ]:
# ── RNN dimensions ────────────────────────────────────────────────────
T         = 10   # sequence length
INPUT_DIM =  4   # input feature size
HIDDEN_DIM = 8   # hidden state size

rng = np.random.default_rng(42)

# Weights (initialised with norm < 1 to trigger vanishing gradient)
W_hh = rng.normal(0, 0.3, (HIDDEN_DIM, HIDDEN_DIM))  # recurrent weight
W_xh = rng.normal(0, 0.3, (HIDDEN_DIM, INPUT_DIM))   # input→hidden weight
b_h  = np.zeros(HIDDEN_DIM)                           # bias

# Synthetic input sequence
x_seq = rng.normal(0, 1, (T, INPUT_DIM))

# ── Forward pass: store hidden states ─────────────────────────────────
def rnn_forward(x_seq: np.ndarray, W_hh: np.ndarray,
                W_xh: np.ndarray, b_h: np.ndarray) -> list:
    """Run RNN forward pass; return list of hidden states."""
    h = np.zeros(HIDDEN_DIM)
    hidden_states = []
    for t in range(len(x_seq)):
        h = np.tanh(W_hh @ h + W_xh @ x_seq[t] + b_h)
        hidden_states.append(h.copy())
    return hidden_states

h_states = rnn_forward(x_seq, W_hh, W_xh, b_h)
print(f'Hidden states computed for T={T} timesteps, dim={HIDDEN_DIM}')

# ── Manual BPTT: compute gradient norm at each timestep ───────────────
# Gradient at last timestep (loss grad w.r.t. h_T)
delta = np.ones(HIDDEN_DIM)  # d_loss / d_h_T = 1 (dummy loss)

grad_norms = []
for t in range(T - 1, -1, -1):
    # Jacobian of tanh: (1 - h_t^2)
    dtanh = 1.0 - h_states[t] ** 2  # [HIDDEN_DIM]
    # Chain rule: delta_t = (W_hh.T @ delta_{t+1}) * dtanh
    delta = (W_hh.T @ delta) * dtanh
    grad_norms.append(np.linalg.norm(delta))

grad_norms_fwd = grad_norms[::-1]  # reorder from t=0 to t=T-1
print('\nGradient norm at each timestep (t=0 earliest, t=9 latest):')
for t, gn in enumerate(grad_norms_fwd):
    bar = '#' * int(gn * 20)
    print(f'  t={t}: {gn:.4f}  {bar}')

# ── Visualise gradient norms ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(range(T), grad_norms_fwd, marker='o', color='crimson', linewidth=2)
ax.set_xlabel('Timestep')
ax.set_ylabel('Gradient norm')
ax.set_title('Vanishing Gradient in Vanilla RNN')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/nlp03_vanishing.png', dpi=80)
plt.close()
print('\nPlot saved to /tmp/nlp03_vanishing.png')

# Confirm exponential decay
ratios = [grad_norms_fwd[t] / (grad_norms_fwd[t+1] + 1e-9)
          for t in range(T - 1)]
mean_ratio = np.mean(ratios)
print(f'\nMean gradient ratio (t / t+1): {mean_ratio:.3f}')
if mean_ratio > 1.0:
    print('Gradient GROWS moving backward (vanishes at early timesteps).')
else:
    print('Gradient decays moving backward — vanishing gradient confirmed.')


## Level 2: Custom LSTM Cell in PyTorch

The LSTM uses four gates to control information flow:
- **Forget gate** f: how much to erase from cell state
- **Input gate**  i: how much new info to write
- **Cell gate** c̃: candidate cell update
- **Output gate** o: how much of cell state to expose

Cell update: `c_t = f*c_{t-1} + i*c̃_t`
Hidden state: `h_t = o * tanh(c_t)`

In [ ]:
class LSTMCell(nn.Module):
    """Manual LSTM cell — demonstrates gating mechanism explicitly."""
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        # Each gate projects concatenated [x_t, h_{t-1}] to hidden_dim
        self.Wf = nn.Linear(input_dim + hidden_dim, hidden_dim)  # forget gate
        self.Wi = nn.Linear(input_dim + hidden_dim, hidden_dim)  # input gate
        self.Wc = nn.Linear(input_dim + hidden_dim, hidden_dim)  # cell gate
        self.Wo = nn.Linear(input_dim + hidden_dim, hidden_dim)  # output gate

    def forward(self, x: torch.Tensor, h: torch.Tensor,
                c: torch.Tensor):
        """Single timestep. Returns (h_new, c_new)."""
        combined = torch.cat([x, h], dim=-1)          # [B, input+hidden]
        f = torch.sigmoid(self.Wf(combined))           # forget: [B, H]
        i = torch.sigmoid(self.Wi(combined))           # input:  [B, H]
        c_tilde = torch.tanh(self.Wc(combined))        # candidate
        o = torch.sigmoid(self.Wo(combined))           # output:  [B, H]
        c_new = f * c + i * c_tilde                    # updated cell state
        h_new = o * torch.tanh(c_new)                  # new hidden state
        return h_new, c_new

class SequencePredictor(nn.Module):
    """Predict next value in an integer sequence (regression)."""
    def __init__(self, input_dim: int, hidden_dim: int,
                 cell_type: str = 'lstm'):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.cell_type  = cell_type
        if cell_type == 'lstm':
            self.cell = LSTMCell(input_dim, hidden_dim)
        elif cell_type == 'rnn':
            self.rnn = nn.RNNCell(input_dim, hidden_dim)
        elif cell_type == 'gru':
            self.gru = nn.GRUCell(input_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x_seq: torch.Tensor) -> torch.Tensor:
        """x_seq: [B, T, 1]. Returns predicted next value [B]."""
        B, T, _ = x_seq.shape
        h = torch.zeros(B, self.hidden_dim, device=x_seq.device)
        c = torch.zeros(B, self.hidden_dim, device=x_seq.device)
        for t in range(T):
            xt = x_seq[:, t, :]  # [B, 1]
            if self.cell_type == 'lstm':
                h, c = self.cell(xt, h, c)
            elif self.cell_type == 'rnn':
                h = self.rnn(xt, h)
            elif self.cell_type == 'gru':
                h = self.gru(xt, h)
        return self.fc(h).squeeze(-1)  # [B]

def make_seq_data(seq_len: int, n_samples: int = 200):
    """Create (input_seq, target) where target = sum(input)."""
    X = np.random.uniform(0, 1, (n_samples, seq_len, 1)).astype(np.float32)
    y = X.sum(axis=1).squeeze()                        # target = sum of seq
    return torch.tensor(X), torch.tensor(y)

def train_predictor(model: nn.Module, X: torch.Tensor,
                    y: torch.Tensor, epochs: int = 100) -> list:
    """Train sequence predictor; return loss history."""
    opt = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()
    losses_hist = []
    for ep in range(epochs):
        opt.zero_grad()
        pred = model(X.to(device))
        loss = criterion(pred, y.to(device))
        loss.backward()
        # Gradient clipping prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        losses_hist.append(loss.item())
    return losses_hist

# ── Compare RNN vs LSTM on different sequence lengths ─────────────────
INPUT_DIM_L2 = 1
HIDDEN_DIM_L2 = 32
results_l2 = {}

for seq_len in [5, 15, 30]:
    X_seq, y_seq = make_seq_data(seq_len, n_samples=200)
    split = 160
    X_tr, X_te = X_seq[:split], X_seq[split:]
    y_tr, y_te = y_seq[:split], y_seq[split:]

    row = {}
    for cell_type in ['rnn', 'lstm', 'gru']:
        m = SequencePredictor(INPUT_DIM_L2, HIDDEN_DIM_L2, cell_type).to(device)
        train_predictor(m, X_tr, y_tr, epochs=100)
        with torch.no_grad():
            pred_te = m(X_te.to(device)).cpu().numpy()
        mse = float(np.mean((pred_te - y_te.numpy()) ** 2))
        row[cell_type] = mse
    results_l2[seq_len] = row
    print(f'seq_len={seq_len:2d}: '
          f'RNN={row["rnn"]:.4f}  '
          f'LSTM={row["lstm"]:.4f}  '
          f'GRU={row["gru"]:.4f}')

print('\nLower MSE is better. LSTM/GRU should outperform RNN for long sequences.')


## Real-World Example 1: Sentiment Classification with LSTM

Build an end-to-end sentiment classifier:
token → integer index → nn.Embedding → LSTM → final hidden state → sigmoid → binary label.

In [ ]:
# ── Synthetic sentiment corpus ───────────────────────────────────────
sent_pos = [
    'the movie was great and enjoyable',
    'i loved the brilliant performances',
    'fantastic story with amazing direction',
    'excellent script and outstanding acting',
    'truly wonderful and delightful experience',
    'the best film i have seen in years',
    'impressive visuals and great sound design',
    'a masterpiece of modern cinema',
    'superb storytelling and brilliant cast',
    'wonderful ending to a great trilogy',
]
sent_neg = [
    'the movie was boring and predictable',
    'i hated the terrible performances',
    'awful story with poor direction',
    'dreadful script and bad acting',
    'truly horrible and disappointing experience',
    'the worst film i have seen this year',
    'pathetic visuals and mediocre sound',
    'a waste of two hours of my life',
    'dull storytelling and bland cast',
    'horrible ending ruins the entire film',
]

sents_all = sent_pos + sent_neg
labels_all = [1] * 10 + [0] * 10

# ── Vocabulary and encoding ───────────────────────────────────────────
vocab_s = sorted({w for s in sents_all for w in s.split()})
vocab_s = ['<PAD>'] + vocab_s       # index 0 = padding
w2i_s = {w: i for i, w in enumerate(vocab_s)}
VOCAB_S = len(vocab_s)
MAX_LEN = 10

def encode_sent(sent: str, w2i: dict, max_len: int) -> list:
    """Encode sentence to fixed-length index list, zero-padded."""
    ids = [w2i.get(w, 0) for w in sent.split()]
    ids = ids[:max_len] + [0] * max(0, max_len - len(ids))
    return ids

X_s = torch.tensor([encode_sent(s, w2i_s, MAX_LEN) for s in sents_all],
                   dtype=torch.long)
y_s = torch.tensor(labels_all, dtype=torch.float32)

class LSTMSentiment(nn.Module):
    """Binary sentiment classifier using LSTM over word embeddings."""
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm  = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc    = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.embed(x)             # [B, T, E]
        _, (h, _) = self.lstm(emb)      # h: [1, B, H]
        logit = self.fc(h.squeeze(0))   # [B, 1]
        return torch.sigmoid(logit).squeeze(-1)  # [B]

EMBED_S, HIDDEN_S = 16, 32
lstm_clf = LSTMSentiment(VOCAB_S, EMBED_S, HIDDEN_S).to(device)
opt_s  = optim.Adam(lstm_clf.parameters(), lr=0.01)
crit_s = nn.BCELoss()

loss_hist_s = []
for ep in range(50):
    opt_s.zero_grad()
    preds = lstm_clf(X_s.to(device))
    loss  = crit_s(preds, y_s.to(device))
    loss.backward()
    opt_s.step()
    loss_hist_s.append(loss.item())

with torch.no_grad():
    preds_final = lstm_clf(X_s.to(device)).cpu().numpy()
binary_preds = (preds_final > 0.5).astype(int)
acc_s = accuracy_score(labels_all, binary_preds)
print(f'LSTM sentiment accuracy: {acc_s:.3f}')
print(f'Final loss: {loss_hist_s[-1]:.4f}')


## Real-World Example 2: Bidirectional LSTM

A bidirectional LSTM runs two LSTMs over the same sequence:
one left-to-right and one right-to-left. Final representation =
`concat(h_fwd[-1], h_bwd[-1])`, giving the model context from both directions.
This is especially beneficial for NER and token-level tasks.

In [ ]:
class BiLSTMClassifier(nn.Module):
    """Bidirectional LSTM for sentence-level classification."""
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # bidirectional=True doubles the output hidden dim
        self.bilstm = nn.LSTM(embed_dim, hidden_dim,
                              batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)  # 2× for bi-directional

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.embed(x)               # [B, T, E]
        _, (h, _) = self.bilstm(emb)      # h: [2, B, H] (fwd + bwd)
        # Concatenate forward and backward final hidden states
        h_cat = torch.cat([h[0], h[1]], dim=-1)  # [B, 2H]
        return torch.sigmoid(self.fc(h_cat)).squeeze(-1)  # [B]

bilstm_clf = BiLSTMClassifier(VOCAB_S, EMBED_S, HIDDEN_S).to(device)
opt_bi  = optim.Adam(bilstm_clf.parameters(), lr=0.01)

loss_hist_bi = []
for ep in range(50):
    opt_bi.zero_grad()
    preds = bilstm_clf(X_s.to(device))
    loss  = crit_s(preds, y_s.to(device))
    loss.backward()
    opt_bi.step()
    loss_hist_bi.append(loss.item())

with torch.no_grad():
    preds_bi = bilstm_clf(X_s.to(device)).cpu().numpy()
acc_bi = accuracy_score(labels_all, (preds_bi > 0.5).astype(int))
print(f'Unidirectional LSTM accuracy: {acc_s:.3f}')
print(f'Bidirectional  LSTM accuracy: {acc_bi:.3f}')

# ── Count params ─────────────────────────────────────────────────────
def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\nUni-LSTM params: {count_params(lstm_clf):,}')
print(f'Bi-LSTM  params: {count_params(bilstm_clf):,}')
print('BiLSTM has ~2x more parameters in the LSTM layer but richer context.')

# ── Visualise loss curves ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(loss_hist_s,  label='Unidirectional LSTM', color='steelblue')
ax.plot(loss_hist_bi, label='Bidirectional LSTM',  color='darkorange')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE loss')
ax.set_title('Uni vs Bidirectional LSTM Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/nlp03_bilstm.png', dpi=80)
plt.close()
print('Loss comparison saved to /tmp/nlp03_bilstm.png')


## Real-World Example 3 + Comparison: Gradient Analysis

### Example 3: Gradient Norms Across Timesteps — RNN vs LSTM vs GRU
Measure how gradient signal is preserved as we backpropagate through
T=20 timesteps for RNN, LSTM, and GRU.

### Comparison: RNN vs LSTM vs GRU Accuracy on Long Sequences (length 20)

In [ ]:
# ── Gradient norm analysis across timesteps ──────────────────────────
SEQ_LEN_GRAD = 20
HIDDEN_G     = 16
INPUT_G      = 4

# Build a small dataset for gradient tracking
X_grad = torch.randn(8, SEQ_LEN_GRAD, INPUT_G, requires_grad=False)
y_grad = torch.randint(0, 2, (8,)).float()

def compute_grad_norms_per_timestep(
        cell_type: str, X: torch.Tensor, y: torch.Tensor,
        hidden_dim: int = 16, input_dim: int = 4) -> list:
    """Forward + backward pass; collect gradient norms at each timestep."""
    B, T, D = X.shape
    h = torch.zeros(B, hidden_dim, requires_grad=True)
    c = torch.zeros(B, hidden_dim)

    if cell_type == 'rnn':
        cell = nn.RNNCell(input_dim, hidden_dim)
    elif cell_type == 'lstm':
        cell = LSTMCell(input_dim, hidden_dim)
    elif cell_type == 'gru':
        cell = nn.GRUCell(input_dim, hidden_dim)

    fc = nn.Linear(hidden_dim, 1)

    # Forward through all timesteps, track intermediate h tensors
    hs = []
    for t in range(T):
        xt = X[:, t, :]
        if cell_type == 'lstm':
            h, c = cell(xt, h, c)
        elif cell_type == 'rnn':
            h = cell(xt, h)
        elif cell_type == 'gru':
            h = cell(xt, h)
        hs.append(h)

    # Compute scalar loss from final hidden state
    logit = fc(hs[-1]).squeeze(-1)
    loss  = nn.BCEWithLogitsLoss()(logit, y)
    loss.backward()

    # Compute gradient norms at each stored hidden state
    grad_norms_ts = []
    for h_t in hs:
        if h_t.grad is not None:
            grad_norms_ts.append(h_t.grad.norm().item())
        else:
            grad_norms_ts.append(0.0)
    return grad_norms_ts

# ── Plot gradient norm profiles ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
colors = {'rnn': 'crimson', 'lstm': 'steelblue', 'gru': 'forestgreen'}
# Use simpler approach: measure grad norms through an unrolled sequence
for cell_type in ['rnn', 'lstm', 'gru']:
    torch.manual_seed(0)
    if cell_type == 'rnn':
        rnn_m = nn.RNN(INPUT_G, HIDDEN_G, batch_first=True)
    elif cell_type == 'lstm':
        rnn_m = nn.LSTM(INPUT_G, HIDDEN_G, batch_first=True)
    elif cell_type == 'gru':
        rnn_m = nn.GRU(INPUT_G, HIDDEN_G, batch_first=True)

    X_t = torch.randn(4, SEQ_LEN_GRAD, INPUT_G)
    X_t.requires_grad_(True)
    outputs, _ = rnn_m(X_t)
    # Loss depends only on last output
    loss_t = outputs[:, -1, :].sum()
    loss_t.backward()
    # Gradient norm at each input timestep proxies gradient flow
    grad_per_t = X_t.grad.norm(dim=-1).mean(dim=0).detach().numpy()
    ax.plot(range(SEQ_LEN_GRAD), grad_per_t,
            label=cell_type.upper(), color=colors[cell_type], linewidth=2)

ax.set_xlabel('Timestep')
ax.set_ylabel('Input gradient norm')
ax.set_title('Gradient Flow: RNN vs LSTM vs GRU (seq_len=20)')
ax.legend()
ax.grid(True, alpha=0.3)

# ── Bar chart: MSE on sequence-length-20 task ─────────────────────────
ax2 = axes[1]
mse_vals = results_l2.get(30, results_l2.get(15, results_l2[5]))
model_names = list(mse_vals.keys())
mse_scores  = list(mse_vals.values())
bar_c = [colors[m] for m in model_names]
bars  = ax2.bar([m.upper() for m in model_names], mse_scores, color=bar_c)
for bar, val in zip(bars, mse_scores):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.001,
             f'{val:.4f}', ha='center', va='bottom', fontsize=9)
ax2.set_ylabel('Test MSE (sequence sum task)')
ax2.set_title('RNN vs LSTM vs GRU — Long-Sequence Accuracy')
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('RNN Architecture Comparison', fontsize=13)
plt.tight_layout()
plt.savefig('/tmp/nlp03_comparison.png', dpi=80, bbox_inches='tight')
plt.close()
print('Comparison plot saved to /tmp/nlp03_comparison.png')

print('\nKey insights:')
print('  1. RNN gradients decay sharply for early timesteps.')
print('  2. LSTM/GRU gates protect gradient flow via additive cell-state update.')
print('  3. GRU is simpler (2 gates) vs LSTM (3 gates + cell state).')
print('  4. Use LSTM/GRU for sequences > 10 tokens; vanilla RNN struggles.')
